In [82]:
import numpy as np
import pandas as pd
import neurokit2 as nk

Imported reshape function as it will be needed to make row-wise descriptives

In [83]:
#Now we make the reshape function
#4 parameters, your dataframe, your window size or how many hz of rows should be in each row, purity for how much each row can tolerate label confounsion, and stride size for how much we should overlap
#By default this is every 10 seconds of data
def reshape_function(df, name = 'eda', window_size = 60, stride_size = 15, hertz=4, purity_threshold=0.8):
    window_row = window_size*hertz
    stride_row = stride_size*hertz
    #Get the subject names
    subjects = sorted(
        list(set(col.split("_")[0] for col in df.columns if "_" in col))
    )
    #Store the chunks of data while we're looping across the time span
    all_window = []
    for i in subjects:
        signal_col = f'{i}_{name}'
        label_col = f'{i}_Label'
        #Kill NAs in individual sequence
        individual_df = df[[signal_col,label_col]].dropna()
        total_rows = len(individual_df)
        #For each subject we take a chunk of their data by our determined window size then stack onto a list (in this case "all window")
        #Chunk is determined through standard range check then we check the purity threshold if we keep the chunk, based on birken's paper design
        for j in range(0,total_rows-window_row+1, stride_row):
            end = j + window_row
            signal_window = individual_df[signal_col].iloc[j:end].values
            label_window = individual_df[label_col].iloc[j:end].values

            # Determine the majority label and calculate if there exists a supermajority in the chunk to determine if kept
            value_counts = pd.Series(label_window).value_counts()
            majority_label = value_counts.index[0]
            majority_count = value_counts.iloc[0]

            purity = majority_count / window_row

            # Retain only windows meeting or exceeding the requested purity ratio
            if purity >= purity_threshold:
                row_data = [i, majority_label] + list(signal_window)
                all_window.append(row_data)

    feature_cols = [f"sample_{i}" for i in range(window_row)]
    columns = ["subject_id", "label"] + feature_cols

    return pd.DataFrame(all_window, columns=columns)

    





I also want visualization boxplots for each feature set similar to Zhu lili's breakdown

Import tonic and phasic files

In [84]:
tonic = pd.read_csv("../Data/wesad_eda/tonic_signal.csv")
phasic = pd.read_csv("../Data/wesad_eda/phasic_signal.csv")

This section is just applying the reshape function and verifying the outputs

In [85]:
tonic_10 = reshape_function(tonic, name='Tonic')

In [86]:
tonic_10.shape

(4655, 242)

In [87]:
tonic_10.head()

,subject_id,label,sample_0,sample_1,sample_2,sample_3,sample_4,sample_5,sample_6,sample_7,...,sample_230,sample_231,sample_232,sample_233,sample_234,sample_235,sample_236,sample_237,sample_238,sample_239
0,S10,0.0,0.012549,0.013171,0.013922,0.014796,0.015787,0.016890,0.018099,0.019409,...,0.099717,0.099668,0.099620,0.099573,0.099526,0.099481,0.099437,0.099394,0.099353,0.099314
1,S10,0.0,0.097403,0.097574,0.097708,0.097810,0.097881,0.097924,0.097942,0.097937,...,0.098489,0.098473,0.098456,0.098441,0.098425,0.098411,0.098397,0.098384,0.098373,0.098362
2,S10,0.0,0.099453,0.099512,0.099565,0.099612,0.099654,0.099690,0.099721,0.099748,...,0.100008,0.100008,0.100001,0.099988,0.099967,0.099939,0.099903,0.099859,0.099807,0.099746
3,S10,1.0,0.099676,0.099596,0.099507,0.099410,0.099306,0.099194,0.099075,0.098951,...,0.104067,0.104058,0.104050,0.104042,0.104036,0.104032,0.104029,0.104028,0.104028,0.104031
4,S10,1.0,0.097387,0.097617,0.097856,0.098105,0.098363,0.098631,0.098907,0.099193,...,0.104607,0.104582,0.104555,0.104528,0.104499,0.104469,0.104439,0.104408,0.104377,0.104345


Tonic descriptives row-wise for max and mean

In [88]:
sample_cols = [col for col in tonic_10.columns if col.startswith('sample_')]
tonic_descriptives = pd.DataFrame()
#Bring Labels into descriptives to the databas
tonic_descriptives['label'] = tonic_10['label']
#Bring subject ID over to descriptives
if 'subject_id' in tonic_10.columns:
    tonic_descriptives['subject_id'] = tonic_10['subject_id']
#Descriptives
tonic_descriptives['tonic_mean'] = tonic_10[sample_cols].mean(axis=1)
tonic_descriptives['tonic_max']  = tonic_10[sample_cols].max(axis=1)

tonic_descriptives.head()

,label,subject_id,tonic_mean,tonic_max
0,0.0,S10,0.089349,0.100753
1,0.0,S10,0.099112,0.100753
2,0.0,S10,0.099471,0.100753
3,1.0,S10,0.104281,0.115908
4,1.0,S10,0.106216,0.115908


In [89]:
phasic_10 = reshape_function(phasic, name='Phasic')

Phasic descriptives row-wise for mean, peak counts and rise time standard deviation

In [90]:
phasic_10.shape

(4655, 242)

In [91]:
sample_cols = [col for col in phasic_10.columns if col.startswith('sample_')]
phasic_descriptives = pd.DataFrame()
#Bring Labels into descriptives to the database
phasic_descriptives['label'] = phasic_10['label']
#Bring subject ID over to descriptives
if 'subject_id' in phasic_10.columns:
    phasic_descriptives['subject_id'] = phasic_10['subject_id']
#Descriptives
phasic_descriptives['phasic_mean'] = phasic_10[sample_cols].mean(axis=1)
phasic_descriptives['peaks_count'] = (phasic_10[sample_cols] > 0.01).sum(axis=1)
phasic_descriptives.head()

,label,subject_id,phasic_mean,peaks_count
0,0.0,S10,0.006057,15
1,0.0,S10,0.000084,0
2,0.0,S10,0.000078,0
3,1.0,S10,0.004718,11
4,1.0,S10,0.002144,4


Test example before I implemented on full dataframe the rise time standard deviation descriptive

In [92]:
row_example = phasic_10.iloc[0:1,2:]
signals, info = nk.eda_peaks(np.array(row_example).flatten(), sampling_rate=4)
# Calculate rise times in seconds
rise_times_sec = (info["SCR_Peaks"] - info["SCR_Onsets"]) / 4
np.nanstd(rise_times_sec)


0.375

In [ ]:
def rise_time_std(row, sampling_rate = 4):
    try:
        #Uses neurokits peak detection to get the onsets then convert it to a time scale and calculate the standard deviation
        signals, info = nk.eda_peaks(row, sampling_rate=sampling_rate)
        rise_times = (info["SCR_Peaks"] - info["SCR_Onsets"]) / sampling_rate
        return np.nanstd(rise_times)
    #There is a lot of rows with no peaks at all, need to decide how to handle the NAs where the std depreciates thereof
    except Exception:
        return np.nan
#Create a new column after apply row wise the above function
phasic_descriptives['phasic_std_rise_time'] = phasic_10[sample_cols].apply(
    lambda row: rise_time_std(row), axis=1
)

phasic_descriptives.head()

/home/sj/.local/lib/python3.12/site-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice encountered
  valid_peaks = np.logical_and(info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"]))  # pylint: disable=E1111
/home/sj/.local/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/sj/.local/lib/python3.12/site-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice encountered
  valid_peaks = np.logical_and(info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"]))  # pylint: disable=E1111
/home/sj/.local/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/sj/.local/lib/python3.12/site-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice

,label,subject_id,phasic_mean,peaks_count,phasic_std_rise_time
0,0.0,S10,0.006057,15,0.375000
1,0.0,S10,0.000084,0,3.659083
2,0.0,S10,0.000078,0,3.000000
3,1.0,S10,0.004718,11,0.353553
4,1.0,S10,0.002144,4,0.000000
...,...,...,...,...,...
4650,0.0,S9,0.001341,5,1.291499
4651,0.0,S9,0.000771,2,0.000000
4652,0.0,S9,0.000640,4,0.273861
4653,0.0,S9,0.003120,11,0.796869


Create and save file descriptive file

In [94]:
feature_dataframe = pd.concat(
    [phasic_descriptives, tonic_descriptives.drop(columns=["subject_id", "label"])],
    axis=1,
)
feature_dataframe.head()

,label,subject_id,phasic_mean,peaks_count,phasic_std_rise_time,tonic_mean,tonic_max
0,0.0,S10,0.006057,15,0.375000,0.089349,0.100753
1,0.0,S10,0.000084,0,3.659083,0.099112,0.100753
2,0.0,S10,0.000078,0,3.000000,0.099471,0.100753
3,1.0,S10,0.004718,11,0.353553,0.104281,0.115908
4,1.0,S10,0.002144,4,0.000000,0.106216,0.115908


In [95]:
feature_dataframe.to_csv("../Data/wesad_eda/feature_dataframe.csv", index=False)
